In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/quora-question-pairs/train.csv.zip
/kaggle/input/quora-question-pairs/sample_submission.csv.zip
/kaggle/input/quora-question-pairs/test.csv
/kaggle/input/quora-question-pairs/test.csv.zip


In [2]:
import zipfile
import pandas as pd
import os

# Extract all zip files
for zip_file in ['train.csv.zip', 'test.csv.zip', 'sample_submission.csv.zip']:
    with zipfile.ZipFile(f"/kaggle/input/quora-question-pairs/{zip_file}", 'r') as zip_ref:
        zip_ref.extractall("./data/")

# Load datasets
train_df = pd.read_csv("./data/train.csv")
test_df = pd.read_csv("./data/test.csv")
sample_submission_df = pd.read_csv("./data/sample_submission.csv")

/tmp/ipykernel_36/1089203271.py:12: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv("./data/test.csv")


In [5]:
!pip -q install sentence-transformers
import zipfile
import pandas as pd
from sklearn import model_selection
from datasets import Dataset
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, BinaryClassificationEvaluator
from sentence_transformers.similarity_functions import SimilarityFunction
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sklearn.metrics import f1_score
import numpy as np
import torch
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.8 MB/s eta 0:00:00:00:0100:01


2025-09-06 18:13:13.716376: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757182393.923393      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757182393.979692      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# Preprocess train data
train_df = train_df.dropna()
train_df = train_df.rename(columns={'is_duplicate': "label"})
train_df = train_df[["question1", "question2", "label"]]

# Create train, validation, and test splits from train data
train_val, internal_test = model_selection.train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df["label"]
)
train, val = model_selection.train_test_split(
    train_val, test_size=0.25, random_state=42, stratify=train_val["label"]
)
train_ds = Dataset.from_pandas(train.reset_index(drop=True))
val_ds = Dataset.from_pandas(val.reset_index(drop=True))
internal_test_ds = Dataset.from_pandas(internal_test.reset_index(drop=True))

# Prepare test data for prediction
test_ds = Dataset.from_pandas(test_df[["question1", "question2"]].reset_index(drop=True))
test_ids = test_df["test_id"].values

NameError: name 'model_selection' is not defined

In [6]:
# Extract dataset
with zipfile.ZipFile("/kaggle/input/quora-question-pairs/train.csv.zip", 'r') as zip_ref:
    zip_ref.extractall("./train/")

# Load and preprocess data
df = pd.read_csv("./train/train.csv")
df = df.dropna()
df = df.rename(columns={'is_duplicate': "label"})
df = df[["question1", "question2", "label"]]

# Create train, validation, and test splits
train_val, test = model_selection.train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
train, val = model_selection.train_test_split(
    train_val, test_size=0.25, random_state=42, stratify=train_val["label"]
)  # 0.25 x 0.8 = 0.2 of original data
train_ds = Dataset.from_pandas(train.reset_index(drop=True))
val_ds = Dataset.from_pandas(val.reset_index(drop=True))
test_ds = Dataset.from_pandas(test.reset_index(drop=True))

# Configuration
config = {
    "model_path": "microsoft/xtremedistil-l6-h256-uncased",
    "learning_rate": 5e-4,
    "train_batch_size": 64,  # Reduced for memory constraints
    "eval_batch_size": 64,
    "epochs": 5,
    "warmup_ratio": 0.1,
    "output_dir": "./output"
}

In [7]:
class BinaryF1Evaluator:
    def __init__(self, sentences1, sentences2, labels, threshold=0.9, name=""):
        self.sentences1 = sentences1
        self.sentences2 = sentences2
        self.labels = labels
        self.threshold = threshold
        self.name = name

    def __call__(self, model):
        # Compute embeddings
        embeddings1 = model.encode(self.sentences1, batch_size=config["eval_batch_size"], convert_to_tensor=True)
        embeddings2 = model.encode(self.sentences2, batch_size=config["eval_batch_size"], convert_to_tensor=True)
        # Compute cosine similarity
        similarities = torch.nn.functional.cosine_similarity(embeddings1, embeddings2).cpu().numpy()
        # Predict duplicates based on threshold
        predictions = (similarities >= self.threshold).astype(int)
        # Compute F1-score
        f1 = f1_score(self.labels, predictions)
        return {f"f1_score/{self.name}": f1}

# Initialize evaluators for validation and test sets
val_f1_evaluator = BinaryF1Evaluator(
    sentences1=val_ds["question1"],
    sentences2=val_ds["question2"],
    labels=val_ds["label"],
    name="val"
)
test_f1_evaluator = BinaryF1Evaluator(
    sentences1=test_ds["question1"],
    sentences2=test_ds["question2"],
    labels=test_ds["label"],
    name="test"
)

In [8]:
# Load model
model = SentenceTransformer(config["model_path"])
# Evaluate on test set
benchmark_f1 = test_f1_evaluator(model)["f1_score/test"]
print(f"Benchmark F1-score: {benchmark_f1:.4f}")


config.json:   0%|          | 0.00/525 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/51.0M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/51.0M [00:00<?, ?B/s]

Batches:   0%|          | 0/1264 [00:00<?, ?it/s]

Batches:   0%|          | 0/1264 [00:00<?, ?it/s]

Benchmark F1-score: 0.5510


In [9]:
# Load model
model = SentenceTransformer(config["model_path"])
train_loss = losses.CosineSimilarityLoss(model=model)

# Training arguments
args = SentenceTransformerTrainingArguments(
    output_dir=os.path.join(config["output_dir"], "biencoder_cosine"),
    num_train_epochs=config["epochs"],
    learning_rate=config["learning_rate"],
    per_device_train_batch_size=config["train_batch_size"],
    per_device_eval_batch_size=config["eval_batch_size"],
    warmup_ratio=config["warmup_ratio"],
    fp16=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)

# Trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=train_loss,
    evaluator=val_f1_evaluator
)
trainer.train()

# Save model
model.save(os.path.join(config["output_dir"], "biencoder_cosine"))

# Evaluate on test set
cosine_f1 = test_f1_evaluator(model)["f1_score/test"]
print(f"Bi-Encoder (Cosine Similarity Loss) F1-score: {cosine_f1:.4f}")

TypeError: SentenceTransformerTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
!pip -q install sentence-transformers
import zipfile
import pandas as pd
from sklearn import model_selection
from datasets import Dataset
from sentence_transformers import SentenceTransformer, losses, CrossEncoder
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.similarity_functions import SimilarityFunction
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sklearn.metrics import f1_score
import numpy as np
import torch
import os

# Extract dataset
with zipfile.ZipFile("/kaggle/input/quora-question-pairs/train.csv.zip", 'r') as zip_ref:
    zip_ref.extractall("./train/")

# Load and preprocess data
df = pd.read_csv("./train/train.csv")
df = df.dropna()
df = df.rename(columns={'is_duplicate': "label"})
df = df[["question1", "question2", "label"]]

# Create train, validation, and test splits
train_val, test = model_selection.train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
train, val = model_selection.train_test_split(
    train_val, test_size=0.25, random_state=42, stratify=train_val["label"]
)
train_ds = Dataset.from_pandas(train.reset_index(drop=True))
val_ds = Dataset.from_pandas(val.reset_index(drop=True))
test_ds = Dataset.from_pandas(test.reset_index(drop=True))

# Configuration
config = {
    "model_path": "microsoft/xtremedistil-l6-h256-uncased",
    "learning_rate": 5e-4,
    "train_batch_size": 64,
    "eval_batch_size": 64,
    "epochs": 5,
    "warmup_ratio": 0.1,
    "output_dir": "./output"
}

# F1 Evaluator
class BinaryF1Evaluator:
    def __init__(self, sentences1, sentences2, labels, threshold=0.9, name=""):
        self.sentences1 = sentences1
        self.sentences2 = sentences2
        self.labels = labels
        self.threshold = threshold
        self.name = name

    def __call__(self, model):
        embeddings1 = model.encode(self.sentences1, batch_size=config["eval_batch_size"], convert_to_tensor=True)
        embeddings2 = model.encode(self.sentences2, batch_size=config["eval_batch_size"], convert_to_tensor=True)
        similarities = torch.nn.functional.cosine_similarity(embeddings1, embeddings2).cpu().numpy()
        predictions = (similarities >= self.threshold).astype(int)
        f1 = f1_score(self.labels, predictions)
        return {f"f1_score/{self.name}": f1}

val_f1_evaluator = BinaryF1Evaluator(val_ds["question1"], val_ds["question2"], val_ds["label"], name="val")
test_f1_evaluator = BinaryF1Evaluator(test_ds["question1"], test_ds["question2"], test_ds["label"], name="test")

# Experiment 1: Benchmark
model = SentenceTransformer(config["model_path"])
benchmark_f1 = test_f1_evaluator(model)["f1_score/test"]
print(f"Benchmark F1-score: {benchmark_f1:.4f}")

# Experiment 2: Bi-Encoder with Cosine Similarity Loss
model = SentenceTransformer(config["model_path"])
train_loss = losses.CosineSimilarityLoss(model=model)
args = SentenceTransformerTrainingArguments(
    output_dir=os.path.join(config["output_dir"], "biencoder_cosine"),
    num_train_epochs=config["epochs"],
    learning_rate=config["learning_rate"],
    per_device_train_batch_size=config["train_batch_size"],
    per_device_eval_batch_size=config["eval_batch_size"],
    warmup_ratio=config["warmup_ratio"],
    fp16=True,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=train_loss,
    evaluator=val_f1_evaluator
)
trainer.train()
model.save(os.path.join(config["output_dir"], "biencoder_cosine"))
cosine_f1 = test_f1_evaluator(model)["f1_score/test"]
print(f"Bi-Encoder (Cosine Similarity Loss) F1-score: {cosine_f1:.4f}")

# Experiment 3: Bi-Encoder with Contrastive Loss
model = SentenceTransformer(config["model_path"])
train_loss = losses.ContrastiveLoss(model=model, margin=0.5)
args = SentenceTransformerTrainingArguments(
    output_dir=os.path.join(config["output_dir"], "biencoder_contrastive"),
    num_train_epochs=config["epochs"],
    learning_rate=config["learning_rate"],
    per_device_train_batch_size=config["train_batch_size"],
    per_device_eval_batch_size=config["eval_batch_size"],
    warmup_ratio=config["warmup_ratio"],
    fp16=True,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=train_loss,
    evaluator=val_f1_evaluator
)
trainer.train()
model.save(os.path.join(config["output_dir"], "biencoder_contrastive"))
contrastive_f1 = test_f1_evaluator(model)["f1_score/test"]
print(f"Bi-Encoder (Contrastive Loss) F1-score: {contrastive_f1:.4f}")

# Experiment 4: Bi-Encoder with Multiple Negatives Ranking Loss
train_positive = train[train["label"] == 1][["question1", "question2"]]
train_positive_ds = Dataset.from_pandas(train_positive.reset_index(drop=True))
model = SentenceTransformer(config["model_path"])
train_loss = losses.MultipleNegativesRankingLoss(model=model)
args = SentenceTransformerTrainingArguments(
    output_dir=os.path.join(config["output_dir"], "biencoder_mnrl"),
    num_train_epochs=config["epochs"],
    learning_rate=config["learning_rate"],
    per_device_train_batch_size=config["train_batch_size"],
    per_device_eval_batch_size=config["eval_batch_size"],
    warmup_ratio=config["warmup_ratio"],
    fp16=True,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_positive_ds,
    eval_dataset=val_ds,
    loss=train_loss,
    evaluator=val_f1_evaluator
)
trainer.train()
model.save(os.path.join(config["output_dir"], "biencoder_mnrl"))
mnrl_f1 = test_f1_evaluator(model)["f1_score/test"]
print(f"Bi-Encoder (Multiple Negatives Ranking Loss) F1-score: {mnrl_f1:.4f}")

# Experiment 5: Cross-Encoder
cross_model = CrossEncoder(config["model_path"], num_labels=1)
train_pairs = [(row["question1"], row["question2"]) for _, row in train.iterrows()]
train_labels = train["label"].values
val_pairs = [(row["question1"], row["question2"]) for _, row in val.iterrows()]
val_labels = val["label"].values
test_pairs = [(row["question1"], row["question2"]) for _, row in test.iterrows()]
test_labels = test["label"].values

class CrossEncoderF1Evaluator:
    def __init__(self, pairs, labels, threshold=0.5, name=""):
        self.pairs = pairs
        self.labels = labels
        self.threshold = threshold
        self.name = name

    def __call__(self, model):
        scores = model.predict(self.pairs)
        predictions = (scores >= self.threshold).astype(int)
        f1 = f1_score(self.labels, predictions)
        return {f"f1_score/{self.name}": f1}

val_cross_evaluator = CrossEncoderF1Evaluator(val_pairs, val_labels, name="val")
test_cross_evaluator = CrossEncoderF1Evaluator(test_pairs, test_labels, name="test")

args = SentenceTransformerTrainingArguments(
    output_dir=os.path.join(config["output_dir"], "crossencoder"),
    num_train_epochs=config["epochs"],
    learning_rate=config["learning_rate"],
    per_device_train_batch_size=config["train_batch_size"],
    per_device_eval_batch_size=config["eval_batch_size"],
    warmup_ratio=config["warmup_ratio"],
    fp16=True,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)
trainer = SentenceTransformerTrainer(
    model=cross_model,
    args=args,
    train_dataset=Dataset.from_dict({"sentence_pair": train_pairs, "label": train_labels}),
    eval_dataset=Dataset.from_dict({"sentence_pair": val_pairs, "label": val_labels}),
    loss=None,
    evaluator=val_cross_evaluator
)
trainer.train()
cross_model.save(os.path.join(config["output_dir"], "crossencoder"))
cross_f1 = test_cross_evaluator(cross_model)["f1_score/test"]
print(f"Cross-Encoder F1-score: {cross_f1:.4f}")

# Compare results
results = {
    "Benchmark": benchmark_f1,
    "Bi-Encoder (Cosine)": cosine_f1,
    "Bi-Encoder (Contrastive)": contrastive_f1,
    "Bi-Encoder (MNRL)": mnrl_f1,
    "Cross-Encoder": cross_f1
}
print("\nF1-Score Comparison:")
for model_name, f1 in results.items():
    print(f"{model_name}: {f1:.4f}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Batches:   0%|          | 0/1264 [00:00<?, ?it/s]

Batches:   0%|          | 0/1264 [00:00<?, ?it/s]

Benchmark F1-score: 0.5510


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.237900
1000,0.185100
1500,0.192100
2000,0.226500
2500,0.319400
3000,0.290600
3500,0.277200
4000,0.273300
4500,0.277500
5000,0.285600
